In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# ==================== STEP 1: LOAD DATA ====================
train_df = pd.read_csv(r"D:\Download\dataset_final\dataset_final\train.csv")
test_df = pd.read_csv(r"D:\Download\dataset_final\dataset_final\test_features.csv")
val_df = pd.read_csv(r"D:\Download\dataset_final\dataset_final\val.csv")  

TARGET = 'premium_risk'
print(f"原始训练集形状: {train_df.shape}")
print(f"原始测试集形状: {test_df.shape}")
print(f"原始验证集形状: {val_df.shape}")
print(f"\n目标分布:\n{train_df[TARGET].value_counts()}")


# ==================== STEP 2: FEATURE CLASSIFICATION ====================
# ─── 2.1 识别并移除泄漏特征 ───
LEAKAGE_FEATURES = ['bureau_risk_index']  # ← 检测结果

# ─── 2.2 识别并移除管理/PII列 ───
PII_COLUMNS = ['customer_key', 'applicant_ref_code']

# ─── 2.3 识别噪声特征 ───
NOISE_FEATURES = [f'noise_feature_{i}' for i in range(1, 6)]

# ─── 2.4 移除上述列 ───
COLS_TO_DROP = LEAKAGE_FEATURES + PII_COLUMNS + NOISE_FEATURES
train_df = train_df.drop(columns=[c for c in COLS_TO_DROP if c in train_df.columns])
test_df = test_df.drop(columns=[c for c in COLS_TO_DROP if c in test_df.columns])
val_df = val_df.drop(columns=[c for c in COLS_TO_DROP if c in val_df.columns])
print(f"\n移除列: {COLS_TO_DROP}")
print(f"清理后特征数: {train_df.shape[1] - 1}")


# ==================== STEP 3: DATA CLEANING ====================
# ─── 3.1 清理 spending_profile (脏值 !@9#%8) ───
train_df['spending_profile'] = train_df['spending_profile'].replace('!@9#%8', np.nan)
test_df['spending_profile'] = test_df['spending_profile'].replace('!@9#%8', np.nan)
val_df['spending_profile'] = val_df['spending_profile'].replace('!@9#%8', np.nan)
# ─── 3.2 清理 employment_sector (脏值 _______) ───
train_df['employment_sector'] = train_df['employment_sector'].replace('_______', np.nan)
test_df['employment_sector'] = test_df['employment_sector'].replace('_______', np.nan)
val_df['employment_sector'] = val_df['employment_sector'].replace('_______', np.nan)
# ─── 3.3 清理 debt_portfolio_quality (脏值 _) ───
train_df['debt_portfolio_quality'] = train_df['debt_portfolio_quality'].replace('_', np.nan)
test_df['debt_portfolio_quality'] = test_df['debt_portfolio_quality'].replace('_', np.nan)
val_df['debt_portfolio_quality'] = val_df['debt_portfolio_quality'].replace('_', np.nan)

# ─── 3.4 清理 minimum_payment_only (脏值 NM) ───
train_df['minimum_payment_only'] = train_df['minimum_payment_only'].replace('NM', np.nan)
test_df['minimum_payment_only'] = test_df['minimum_payment_only'].replace('NM', np.nan)
val_df['minimum_payment_only'] = val_df['minimum_payment_only'].replace('NM', np.nan)
# ─── 3.5 解析 account_tenure → 提取月数 ───
def parse_tenure(t):
    """将 '16 Years and 10 Months' 转为总月数"""
    if pd.isna(t) or not isinstance(t, str):
        return np.nan
    years = months = 0
    parts = t.lower().replace('and', ',').split(',')
    for part in parts:
        part = part.strip()
        if 'year' in part:
            try: years = int(''.join(c for c in part if c.isdigit()))
            except: pass
        elif 'month' in part:
            try: months = int(''.join(c for c in part if c.isdigit()))
            except: pass
    return years * 12 + months if (years + months) > 0 else np.nan

train_df['account_tenure_months'] = train_df['account_tenure'].apply(parse_tenure)
test_df['account_tenure_months'] = test_df['account_tenure'].apply(parse_tenure)
val_df['account_tenure_months'] = val_df['account_tenure'].apply(parse_tenure)
train_df = train_df.drop(columns=['account_tenure'])
test_df = test_df.drop(columns=['account_tenure'])
val_df = val_df.drop(columns=['account_tenure'])

# ─── 3.6 将 minimum_payment_only 映射为数值 ───
pay_map = {'Yes': 1, 'No': 0}
train_df['minimum_payment_only'] = train_df['minimum_payment_only'].map(pay_map)
test_df['minimum_payment_only'] = test_df['minimum_payment_only'].map(pay_map)
val_df['minimum_payment_only'] = val_df['minimum_payment_only'].map(pay_map)

# ─── 3.7 将 application_month 转为有序编码 ───
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
month_map = {m: i+1 for i, m in enumerate(month_order)}
train_df['application_month'] = train_df['application_month'].map(month_map)
test_df['application_month'] = test_df['application_month'].map(month_map)
val_df['application_month'] = val_df['application_month'].map(month_map)

# ─── 3.8 处理 prior_debt_products (文本 → 特征数量) ───
def count_debt_products(text):
    if pd.isna(text) or not isinstance(text, str):
        return 0
    items = [x.strip() for x in text.replace(' and ',',').split(',') if x.strip()]
    # 排除 Not Specified
    return sum(1 for item in items if item not in ['Not Specified', ''])

train_df['debt_product_count'] = train_df['prior_debt_products'].apply(count_debt_products)
test_df['debt_product_count'] = test_df['prior_debt_products'].apply(count_debt_products)
val_df['debt_product_count'] = val_df['prior_debt_products'].apply(count_debt_products)

# 提取债务类型是否有高风险产品
def has_high_risk_debt(text):
    if pd.isna(text) or not isinstance(text, str):
        return 0
    text_lower = text.lower()
    risky = ['payday loan', 'credit-builder loan']
    return int(any(r in text_lower for r in risky))

train_df['has_risky_debt'] = train_df['prior_debt_products'].apply(has_high_risk_debt)
test_df['has_risky_debt'] = test_df['prior_debt_products'].apply(has_high_risk_debt)
val_df['has_risky_debt'] = val_df['prior_debt_products'].apply(has_high_risk_debt)

# 移除原始文本列
train_df = train_df.drop(columns=['prior_debt_products'])
test_df = test_df.drop(columns=['prior_debt_products'])
val_df = val_df.drop(columns=['prior_debt_products'])
# ─── 3.9 处理极端异常值（数值列的极端值 → Winsorize） ───
numeric_candidates = train_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_candidates = [c for c in numeric_candidates if c != TARGET]

def winsorize_series(s, lower=0.01, upper=0.99):
    q_low = s.quantile(lower)
    q_high = s.quantile(upper)
    return s.clip(q_low, q_high)

for col in numeric_candidates:
    train_df[col] = winsorize_series(train_df[col])
    if col in test_df.columns:
        # 使用训练集的边界
        q_low = train_df[col].quantile(0.01)
        q_high = train_df[col].quantile(0.99)
        test_df[col] = pd.to_numeric(test_df[col], errors='coerce').clip(q_low, q_high)
    if col in val_df.columns:
        # 使用训练集的边界
        q_low = train_df[col].quantile(0.01)
        q_high = train_df[col].quantile(0.99)
        val_df[col] = pd.to_numeric(val_df[col], errors='coerce').clip(q_low, q_high)


# ==================== STEP 4: FEATURE / TARGET SPLIT ====================
y = train_df[TARGET]
X = train_df.drop(columns=[TARGET])

# # 80/20 分割为训练/验证集 (分层采样)
# X_train, X_val, y_train, y_val = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )
# print(f"\n训练集: {X_train.shape}, 验证集: {X_val.shape}")
# print(f"训练目标分布:\n{y_train.value_counts()}")
# print(f"\n验证目标分布:\n{y_val.value_counts()}")

# 假设你已经有了 train_df 和 val_df（或你自己命名的验证集变量）
# 那么直接使用现有的数据集，而不是重新分割

# 如果你的验证集变量名不同，请相应调整
y_train = train_df[TARGET]
X_train = train_df.drop(columns=[TARGET])

y_val = val_df[TARGET]  # 假设这是你现有的验证集标签
X_val = val_df.drop(columns=[TARGET])  # 假设这是你现有的验证集特征

print(f"\n训练集: {X_train.shape}, 验证集: {X_val.shape}")
print(f"训练目标分布:\n{y_train.value_counts()}")
print(f"\n验证目标分布:\n{y_val.value_counts()}")
# ==================== STEP 5: DEFINE COLUMN GROUPS ====================
NUMERIC_COLS = X_train.select_dtypes(include=[np.number]).columns.tolist()
CATEGORICAL_COLS = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"\n数值特征 ({len(NUMERIC_COLS)}): {NUMERIC_COLS}")
print(f"分类特征 ({len(CATEGORICAL_COLS)}): {CATEGORICAL_COLS}")


# ==================== STEP 6: BUILD PREPROCESSING PIPELINE ====================
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NUMERIC_COLS),
        ('cat', categorical_transformer, CATEGORICAL_COLS)
    ],
    remainder='drop'
)


# ==================== STEP 7: BASELINE MODEL PIPELINE ====================
# 使用 Random Forest 作为基线模型（天然支持多类和不平衡）
baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=5,
        class_weight='balanced',   # 处理类别不平衡
        random_state=42,
        n_jobs=-1
    ))
])

# 训练
baseline_model.fit(X_train, y_train)

# 预测
y_pred = baseline_model.predict(X_val)


# ==================== STEP 8: VALIDATION RESULTS ====================
acc = accuracy_score(y_val, y_pred)
macro_f1 = f1_score(y_val, y_pred, average='macro')
print(f"\n{'='*50}")
print(f"  📊 BASELINE MODEL VALIDATION RESULTS")
print(f"{'='*50}")
print(f"  Model: Random Forest (balanced, 200 trees)")
print(f"  Accuracy:  {acc:.4f}")
print(f"  Macro-F1:  {macro_f1:.4f}")
print(f"\n  Classification Report:")
print(classification_report(y_val, y_pred, digits=4))

print(f"\n  Confusion Matrix:")
cm = confusion_matrix(y_val, y_pred)
labels = sorted(y_val.unique())
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df.to_string())

# ─── Cross-validation 估计（更稳健） ───
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=5,
        class_weight='balanced', random_state=42, n_jobs=-1
    ))
])
cv_acc = cross_val_score(cv_pipe, X, y, cv=cv, scoring='accuracy')
cv_f1 = cross_val_score(cv_pipe, X, y, cv=cv, scoring='f1_macro')
print(f"\n  5-Fold Cross-Validation (full training set):")
print(f"  Accuracy:  {cv_acc.mean():.4f} ± {cv_acc.std():.4f}")
print(f"  Macro-F1:  {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")


# ==================== STEP 9: GENERATE TEST PREDICTIONS ====================
# 确保测试集特征列与训练集一致
test_features = test_df[[c for c in X.columns if c in test_df.columns]]
# 对齐列
for col in X.columns:
    if col not in test_features.columns:
        test_features[col] = np.nan
test_features = test_features[X.columns]

test_preds = baseline_model.predict(test_features)

submission = pd.DataFrame({
    'applicant_id': test_df['applicant_id'],
    'premium_risk': test_preds
})
submission.to_csv('baseline_predictions.csv', index=False)
print(f"\n  ✅ 测试预测已保存: baseline_predictions.csv ({len(submission)} 行)")
print(f"  预测分布:\n{submission['premium_risk'].value_counts()}")


原始训练集形状: (74375, 33)
原始测试集形状: (12500, 32)
原始验证集形状: (13125, 33)

目标分布:
premium_risk
Standard    39686
High        21586
Low         13103
Name: count, dtype: int64

移除列: ['bureau_risk_index', 'customer_key', 'applicant_ref_code', 'noise_feature_1', 'noise_feature_2', 'noise_feature_3', 'noise_feature_4', 'noise_feature_5']
清理后特征数: 24

训练集: (74375, 25), 验证集: (13125, 25)
训练目标分布:
premium_risk
Standard    39686
High        21586
Low         13103
Name: count, dtype: int64

验证目标分布:
premium_risk
Standard    7003
High        3810
Low         2312
Name: count, dtype: int64

数值特征 (21): ['application_month', 'applicant_age', 'gross_annual_income_gbp', 'net_monthly_income_gbp', 'bank_accounts_held', 'credit_cards_held', 'avg_interest_rate_pct', 'active_loans_count', 'avg_payment_delay_days', 'late_payment_count', 'credit_limit_change_pct', 'credit_inquiry_count', 'total_outstanding_debt_gbp', 'credit_utilisation_pct', 'minimum_payment_only', 'monthly_emi_total_gbp', 'monthly_investment_gbp', 'end_

MemoryError: Unable to allocate 41.2 GiB for an array with shape (74375, 74402) and data type float64

In [7]:
y_train.shape

(74375,)